In [23]:
# ==============================
# Import Libraries
# ==============================
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OrdinalEncoder, PowerTransformer, StandardScaler

In [10]:
# ==============================
# 1. Load Dataset
# ==============================
df = pd.read_csv("customer_credit_risk_dataset_500_rows.csv")

print("Dataset Loaded\n")
print(df.head(), "\n")

Dataset Loaded

  customer_id  age  gender region education_level employment_type  \
0    CUST1000   56   Other  North       Secondary             NaN   
1    CUST1001   69  Female  South        Graduate        Salaried   
2    CUST1002   46  Female   East         Primary        Salaried   
3    CUST1003   32  Female  South        Graduate   Self-Employed   
4    CUST1004   60    Male  South         Primary   Self-Employed   

   annual_income  loan_amount loan_purpose  credit_score  repayment_history  \
0       59117.76     26380.51         Home           729                 11   
1       93300.05     25008.44    Education           733                  1   
2       37129.64      1989.42     Business           703                 12   
3       68556.80     14573.26    Education           733                  7   
4       51140.26     12122.16     Business           411                  8   

   transaction_count  spending_ratio                   join_date  default_flag  
0            

In [11]:
# ==============================
# 2. Dataset Info
# ==============================
print("Dataset Info:\n")
print(df.info(), "\n")

print("Missing Values:\n")
print(df.isnull().sum(), "\n")

Dataset Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        500 non-null    object 
 1   age                500 non-null    int64  
 2   gender             487 non-null    object 
 3   region             500 non-null    object 
 4   education_level    500 non-null    object 
 5   employment_type    467 non-null    object 
 6   annual_income      500 non-null    float64
 7   loan_amount        500 non-null    float64
 8   loan_purpose       500 non-null    object 
 9   credit_score       500 non-null    int64  
 10  repayment_history  500 non-null    int64  
 11  transaction_count  500 non-null    int64  
 12  spending_ratio     500 non-null    float64
 13  join_date          500 non-null    object 
 14  default_flag       500 non-null    int64  
dtypes: float64(3), int64(5), object(7)
memory usage: 58.7+ KB
N

In [12]:
# ==============================
# 3. Date Handling
# ==============================
df['join_date'] = pd.to_datetime(df['join_date'])

df['year'] = df['join_date'].dt.year
df['month'] = df['join_date'].dt.month
df['day'] = df['join_date'].dt.day
df['weekday'] = df['join_date'].dt.weekday

df.drop('join_date', axis=1, inplace=True)

print("Date Features Extracted:\n")
print(df[['year','month','day','weekday']].head(), "\n")

Date Features Extracted:

   year  month  day  weekday
0  2023      7   10        0
1  2017      8    7        0
2  2016      8   18        3
3  2024      8   31        5
4  2019      6    8        5 



In [13]:
# ==============================
# 4. Missing Value Imputation
# ==============================
num_cols = ['age', 'annual_income', 'loan_amount', 'credit_score']
cat_cols = ['gender', 'employment_type']

num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("After Imputation Missing Values:\n")
print(df.isnull().sum(), "\n")

After Imputation Missing Values:

customer_id          0
age                  0
gender               0
region               0
education_level      0
employment_type      0
annual_income        0
loan_amount          0
loan_purpose         0
credit_score         0
repayment_history    0
transaction_count    0
spending_ratio       0
default_flag         0
year                 0
month                0
day                  0
weekday              0
dtype: int64 



In [14]:

# ==============================
# 5. KNN Imputation
# ==============================
knn_cols = ['annual_income', 'loan_amount', 'credit_score']

knn = KNNImputer(n_neighbors=5)
df[knn_cols] = knn.fit_transform(df[knn_cols])

print("After KNN Imputation:\n")
print(df[knn_cols].head(), "\n")

After KNN Imputation:

   annual_income  loan_amount  credit_score
0       59117.76     26380.51         729.0
1       93300.05     25008.44         733.0
2       37129.64      1989.42         703.0
3       68556.80     14573.26         733.0
4       51140.26     12122.16         411.0 



In [15]:
# ==============================
# 6. Outlier Handling (IQR)
# ==============================
def handle_outliers(col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] > upper, upper, df[col])
    df[col] = np.where(df[col] < lower, lower, df[col])

for col in ['annual_income', 'loan_amount', 'credit_score']:
    handle_outliers(col)

print("After Outlier Handling:\n")
print(df[['annual_income','loan_amount','credit_score']].describe(), "\n")

After Outlier Handling:

       annual_income   loan_amount  credit_score
count     500.000000    500.000000    500.000000
mean    51414.832317  19997.515750    570.140000
std     19402.468378   9492.604516    155.622238
min      5000.000000   1000.000000    300.000000
25%     38291.515000  13504.642500    441.500000
50%     50455.310000  20008.665000    563.500000
75%     64746.712500  26609.155000    707.250000
max    104429.508750  46265.923750    850.000000 



In [16]:
# ==============================
# 7. Encoding
# ==============================

# Ordinal Encoding
edu_order = ['Primary', 'Secondary', 'Graduate', 'Post-Graduate']
ord_enc = OrdinalEncoder(categories=[edu_order])

df['education_level'] = ord_enc.fit_transform(df[['education_level']])

# Label Encoding
df['gender'] = df['gender'].map({'Male':0, 'Female':1, 'Other':2})

# One Hot Encoding
df = pd.get_dummies(df, columns=['region','loan_purpose'], drop_first=True)

print("After Encoding:\n")
print(df.head(), "\n")

After Encoding:

  customer_id   age  gender  education_level employment_type  annual_income  \
0    CUST1000  56.0       2              1.0        Salaried       59117.76   
1    CUST1001  69.0       1              2.0        Salaried       93300.05   
2    CUST1002  46.0       1              0.0        Salaried       37129.64   
3    CUST1003  32.0       1              2.0   Self-Employed       68556.80   
4    CUST1004  60.0       0              0.0   Self-Employed       51140.26   

   loan_amount  credit_score  repayment_history  transaction_count  ...  \
0     26380.51         729.0                 11                 23  ...   
1     25008.44         733.0                  1                 28  ...   
2      1989.42         703.0                 12                 57  ...   
3     14573.26         733.0                  7                 26  ...   
4     12122.16         411.0                  8                 25  ...   

   month  day  weekday  region_North  region_South  regio

In [17]:
# ==============================
# 8. Binning
# ==============================

df['income_bin'] = pd.qcut(df['annual_income'], q=4, labels=False)
df['repayment_bin'] = pd.cut(df['repayment_history'], bins=3, labels=False)

print("Binning Output:\n")
print(df[['income_bin','repayment_bin']].head(), "\n")

Binning Output:

   income_bin  repayment_bin
0           2              2
1           3              0
2           0              2
3           3              1
4           2              1 



In [18]:
# ==============================
# 9. Transaction Binning
# ==============================
df['transaction_bin'] = pd.qcut(df['transaction_count'], q=4, labels=False)

print("Transaction Binning:\n")
print(df[['transaction_count','transaction_bin']].head(), "\n")

Transaction Binning:

   transaction_count  transaction_bin
0                 23                0
1                 28                1
2                 57                2
3                 26                1
4                 25                1 



In [19]:
# ==============================
# 10. Feature Engineering
# ==============================

df['debt_to_income'] = df['loan_amount'] / df['annual_income']

print("New Feature (DTI):\n")
print(df[['debt_to_income']].head(), "\n")

New Feature (DTI):

   debt_to_income
0        0.446237
1        0.268043
2        0.053580
3        0.212572
4        0.237038 



In [20]:
# ==============================
# 11. Transformations
# ==============================

df['log_spending'] = np.log1p(df['spending_ratio'])
df['sqrt_spending'] = np.sqrt(df['spending_ratio'])

pt = PowerTransformer(method='yeo-johnson')
df[['loan_amount','annual_income']] = pt.fit_transform(
    df[['loan_amount','annual_income']]
)

print("After Transformations:\n")
print(df[['log_spending','sqrt_spending']].head(), "\n")

After Transformations:

   log_spending  sqrt_spending
0      4.430102       9.107140
1      2.654649       3.635932
2      3.950859       7.140028
3      4.181134       8.027453
4      4.448984       9.194564 



In [21]:
# ==============================
# 12. Scaling
# ==============================

scaler = StandardScaler()
df[['loan_amount','annual_income']] = scaler.fit_transform(
    df[['loan_amount','annual_income']]
)

print("After Scaling:\n")
print(df[['loan_amount','annual_income']].head(), "\n")

After Scaling:

   loan_amount  annual_income
0     0.685193       0.408668
1     0.548936       2.108597
2    -2.045938      -0.725022
3    -0.531245       0.884347
4    -0.800033       0.001991 



In [22]:
# ==============================
# 13. Final Dataset
# ==============================

print("Final Dataset Shape:", df.shape)
print(df.head())

# Save file
df.to_csv("final_processed_dataset.csv", index=False)

print("\nDataset Saved Successfully")

Final Dataset Shape: (500, 29)
  customer_id   age  gender  education_level employment_type  annual_income  \
0    CUST1000  56.0       2              1.0        Salaried       0.408668   
1    CUST1001  69.0       1              2.0        Salaried       2.108597   
2    CUST1002  46.0       1              0.0        Salaried      -0.725022   
3    CUST1003  32.0       1              2.0   Self-Employed       0.884347   
4    CUST1004  60.0       0              0.0   Self-Employed       0.001991   

   loan_amount  credit_score  repayment_history  transaction_count  ...  \
0     0.685193         729.0                 11                 23  ...   
1     0.548936         733.0                  1                 28  ...   
2    -2.045938         703.0                 12                 57  ...   
3    -0.531245         733.0                  7                 26  ...   
4    -0.800033         411.0                  8                 25  ...   

   loan_purpose_Car  loan_purpose_Education